# Hyperparameter Tuning for HuggingFace Models
*Training Phase: Optimizing Model Performance*

This notebook covers systematic approaches to hyperparameter tuning for HuggingFace models, including grid search, random search, and Bayesian optimization techniques.

## Learning Objectives
- Understand the importance of hyperparameter tuning
- Implement grid search and random search
- Use Bayesian optimization with Optuna
- Evaluate and compare tuning strategies
- Apply tuning to real fine-tuning tasks

## Prerequisites
- Basic understanding of machine learning and fine-tuning
- Familiarity with HuggingFace Transformers
- Python libraries: transformers, datasets, optuna

## Setup and Environment

In [ ]:
# Install required packages
# pip install transformers datasets optuna

import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from datasets import load_dataset
import optuna
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Understanding Hyperparameter Tuning

Hyperparameter tuning is crucial for optimizing model performance. Key hyperparameters in HuggingFace include learning rate, batch size, epochs, and model-specific parameters.

In [ ]:
# Key hyperparameters for fine-tuning
HYPERPARAMETERS = {
    "learning_rate": {
        "description": "Step size for gradient descent",
        "typical_range": "1e-5 to 5e-4",
        "impact": "Too high: unstable training; Too low: slow convergence"
    },
    "batch_size": {
        "description": "Number of samples per training step",
        "typical_range": "8 to 64",
        "impact": "Larger: faster training, less stable; Smaller: slower, more stable"
    },
    "num_train_epochs": {
        "description": "Complete passes through training data",
        "typical_range": "2 to 10",
        "impact": "More epochs: better learning, risk of overfitting"
    },
    "weight_decay": {
        "description": "L2 regularization strength",
        "typical_range": "0.0 to 0.1",
        "impact": "Prevents overfitting by penalizing large weights"
    },
    "warmup_steps": {
        "description": "Learning rate warmup period",
        "typical_range": "0 to 500",
        "impact": "Stabilizes training at the beginning"
    }
}

print("🔧 Key Hyperparameters for Fine-tuning:")
for param, details in HYPERPARAMETERS.items():
    print(f"\n{param.upper()}:")
    print(f"  {details['description']}")
    print(f"  Range: {details['typical_range']}")
    print(f"  Impact: {details['impact']}")

## Dataset Preparation

In [ ]:
# Load a dataset for demonstration
print("📚 Loading IMDB dataset for sentiment classification...")
dataset = load_dataset("imdb")

# Use a small subset for faster experimentation
small_train = dataset["train"].select(range(1000))
small_test = dataset["test"].select(range(200))

print(f"Train samples: {len(small_train)}")
print(f"Test samples: {len(small_test)}")

# Load tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=256)

# Tokenize datasets
print("🔤 Tokenizing datasets...")
tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_test = small_test.map(tokenize_function, batched=True)

# Remove text column and set format
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

print("✅ Dataset preparation complete")

## Method 1: Grid Search

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {"accuracy": accuracy, "f1": f1}

def train_with_params(learning_rate, batch_size, num_epochs, weight_decay):
    """Train model with specific hyperparameters"""
    
    # Load fresh model for each trial
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    
    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )
    
    trainer.train()
    eval_results = trainer.evaluate()
    
    return eval_results["eval_f1"]

# Grid search parameters
learning_rates = [1e-5, 5e-5, 1e-4]
batch_sizes = [8, 16]
num_epochs_list = [2, 3]
weight_decays = [0.0, 0.01]

print("🔍 Performing Grid Search...")
grid_results = []

for lr in learning_rates:
    for bs in batch_sizes:
        for epochs in num_epochs_list:
            for wd in weight_decays:
                print(f"Testing: LR={lr}, BS={bs}, Epochs={epochs}, WD={wd}")
                try:
                    f1_score = train_with_params(lr, bs, epochs, wd)
                    grid_results.append({
                        "learning_rate": lr,
                        "batch_size": bs,
                        "num_epochs": epochs,
                        "weight_decay": wd,
                        "f1_score": f1_score
                    })
                    print(f"  F1 Score: {f1_score:.4f}")
                except Exception as e:
                    print(f"  Error: {e}")

# Display results
grid_df = pd.DataFrame(grid_results)
print("\n📊 Grid Search Results:")
print(grid_df.sort_values("f1_score", ascending=False).head())

## Method 2: Random Search

In [ ]:
# Random search with more parameter combinations
np.random.seed(42)
n_trials = 10

print("🎲 Performing Random Search...")

random_results = []

for i in range(n_trials):
    # Randomly sample parameters
    lr = np.random.choice([1e-5, 2e-5, 5e-5, 1e-4, 2e-4])
    bs = np.random.choice([4, 8, 16, 32])
    epochs = np.random.choice([2, 3, 4])
    wd = np.random.choice([0.0, 0.001, 0.01, 0.1])
    
    print(f"Trial {i+1}: LR={lr}, BS={bs}, Epochs={epochs}, WD={wd}")
    try:
        f1_score = train_with_params(lr, bs, epochs, wd)
        random_results.append({
            "learning_rate": lr,
            "batch_size": bs,
            "num_epochs": epochs,
            "weight_decay": wd,
            "f1_score": f1_score
        })
        print(f"  F1 Score: {f1_score:.4f}")
    except Exception as e:
        print(f"  Error: {e}")

# Compare grid vs random
random_df = pd.DataFrame(random_results)
print("\n📊 Random Search Results:")
print(random_df.sort_values("f1_score", ascending=False).head())

print(f"\nGrid Search Best F1: {grid_df['f1_score'].max():.4f}")
print(f"Random Search Best F1: {random_df['f1_score'].max():.4f}")

## Method 3: Bayesian Optimization with Optuna

In [ ]:
def objective(trial):
    """Objective function for Optuna optimization"""
    
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16, 32])
    num_epochs = trial.suggest_int("num_epochs", 2, 5)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    
    print(f"Trial {trial.number}: LR={learning_rate:.2e}, BS={batch_size}, Epochs={num_epochs}, WD={weight_decay}")
    
    try:
        f1_score = train_with_params(learning_rate, batch_size, num_epochs, weight_decay)
        print(f"  F1 Score: {f1_score:.4f}")
        return f1_score
    except Exception as e:
        print(f"  Error: {e}")
        return 0.0  # Return poor score on failure

# Create Optuna study
print("🎯 Starting Bayesian Optimization with Optuna...")
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

# Display results
print("\n📊 Bayesian Optimization Results:")
print(f"Best F1 Score: {study.best_value:.4f}")
print("Best Parameters:")
for param, value in study.best_params.items():
    print(f"  {param}: {value}")

# Plot optimization history (if plotly available)
try:
    import plotly
    fig = optuna.visualization.plot_optimization_history(study)
    fig.show()
except ImportError:
    print("Install plotly for visualization: pip install plotly")

## Comparison of Tuning Methods

In [ ]:
# Compare all methods
methods_comparison = {
    "Grid Search": {
        "trials": len(grid_df),
        "best_f1": grid_df["f1_score"].max(),
        "efficiency": "Low (tests all combinations)"
    },
    "Random Search": {
        "trials": len(random_df),
        "best_f1": random_df["f1_score"].max(),
        "efficiency": "Medium (random sampling)"
    },
    "Bayesian Optimization": {
        "trials": len(study.trials),
        "best_f1": study.best_value,
        "efficiency": "High (intelligent sampling)"
    }
}

print("⚖️ Comparison of Hyperparameter Tuning Methods:")
for method, stats in methods_comparison.items():
    print(f"\n{method}:")
    print(f"  Trials: {stats['trials']}")
    print(f"  Best F1: {stats['best_f1']:.4f}")
    print(f"  Efficiency: {stats['efficiency']}")

## Best Practices for Hyperparameter Tuning

In [ ]:
TUNING_BEST_PRACTICES = {
    "start_simple": [
        "Begin with default hyperparameters",
        "Tune one parameter at a time initially",
        "Use small datasets for initial experiments"
    ],
    "method_selection": [
        "Grid search: Small parameter spaces",
        "Random search: Large parameter spaces",
        "Bayesian: Expensive objective functions"
    ],
    "evaluation": [
        "Use cross-validation when possible",
        "Monitor validation metrics closely",
        "Save best models automatically"
    ],
    "resources": [
        "Use early stopping to save time",
        "Leverage GPU acceleration",
        "Consider distributed training for large models"
    ],
    "common_mistakes": [
        "Over-tuning on small datasets",
        "Ignoring computational costs",
        "Not reproducing results"
    ]
}

print("💡 Hyperparameter Tuning Best Practices:")
for category, practices in TUNING_BEST_PRACTICES.items():
    print(f"\n{category.upper().replace('_', ' ')}:")
    for practice in practices:
        print(f"  ✓ {practice}")

## Key Takeaways

1. **Systematic Approach**: Use structured methods rather than random guessing
2. **Efficiency Matters**: Bayesian optimization often finds good parameters with fewer trials
3. **Validation**: Always evaluate on held-out data to prevent overfitting
4. **Resources**: Consider computational costs and time constraints
5. **Iteration**: Tuning is iterative - start simple and refine

## Next Steps
- Apply these techniques to your specific use cases
- Experiment with different objective metrics
- Consider automated ML platforms for larger-scale tuning
- Learn about advanced techniques like population-based training